In [1]:
import numpy as np
import pandas as pd

In [2]:
SEED_FILE = "Synthetic Dataset.csv"
OUTPUT_FILE = "my_synthetic_students_data.csv"
N_ROWS = 500          # how many fake students to create
RANDOM_SEED = 42      # same seed = same data every time
PASS_MARK = 50        # exam_score >= 50 means Pass

# Numeric columns the code will learn from.
# Internet_quality is NOT here because it is a category (Poor/Average/Good).
NUM_COLS = ["Age", "Study Hours", "Attendance_Percentage", "Total_Screen_Time",
            "Sleep_hours", "Exam_Score"]

FINAL_COLUMNS = ["Student ID", "Age", "Study Hours", "Attendance_Percentage", "Total_Screen_Time",
                 "Sleep_hours", "Internet_Quality", "Exam_Score", "Result"]

In [3]:
PASS_MARK = 50
seed_df = pd.read_csv(SEED_FILE)
seed_df.columns = seed_df.columns.str.strip()
print(seed_df.head(5))

  Student ID  Age  Study Hours  Attendance_Percentage  Total_Screen_Time  \
0       S001   25         12.0                     50                  8   
1       S002   18          2.0                     15                 15   
2       S003   26          4.0                     30                 10   
3       S004   20         10.0                     65                 10   
4       S005   19          5.0                     30                 17   

   Sleep_hours Internet_Quality  Exam_Score Result  
0            7             Good        80.0   Pass  
1            4             Good        25.5   Fail  
2            4             Good        30.5   Fail  
3            6             Poor        50.0   Pass  
4            3          Average        45.5   Fail  


In [4]:
num_cols = ["Age", "Study Hours", "Attendance_Percentage", "Total_Screen_Time",
            "Sleep_hours", "Exam_Score"]

In [5]:
mean = seed_df[num_cols].mean()
cov = seed_df[num_cols].cov()

In [6]:
rng = np.random.default_rng(RANDOM_SEED)
new_rows = rng.multivariate_normal(mean, cov, size=N_ROWS)
df = pd.DataFrame(new_rows, columns=num_cols)

In [7]:
df["Age"] = df["Age"].clip(17, 45).round().astype(int)
df["Study Hours"] = df["Study Hours"].clip(0, 40).round(1)
df["Attendance_Percentage"] = df["Attendance_Percentage"].clip(30, 100).round(1)
df["Total_Screen_Time"] = df["Total_Screen_Time"].clip(1, 16).round(1)
df["Sleep_hours"] = df["Sleep_hours"].clip(3, 10).round(1)
df["Exam_Score"] = df["Exam_Score"].clip(0, 100).round(1)

In [8]:
quality = seed_df["Internet_Quality"].astype(str).str.strip().str.title()
proportions = quality.value_counts(normalize=True)
df["Internet_Quality"] = rng.choice(
    proportions.index, size=len(df), p=proportions.values)

In [9]:
# 4. Result comes from the threshold, same as before
df["Result"] = np.where(df["Exam_Score"] >= PASS_MARK, "Pass", "Fail")

In [10]:
df.insert(0, "Student ID", [f"S{i:04d}" for i in range(1, len(df) + 1)])
df = df[FINAL_COLUMNS]

df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(df)} rows to {OUTPUT_FILE}\n")
print(df.head(10).to_string(index=False))
print(f"\nPass rate: {(df['Result'] == 'Pass').mean():.0%}")
print("\nInternet Quality split:")
print(df["Internet_Quality"].value_counts(normalize=True).round(2).to_string())

Saved 500 rows to my_synthetic_students_data.csv

Student ID  Age  Study Hours  Attendance_Percentage  Total_Screen_Time  Sleep_hours Internet_Quality  Exam_Score Result
     S0001   24          2.2                   48.1                8.1          8.1             Poor        39.8   Fail
     S0002   19          7.0                   46.5               10.4          4.1             Poor        51.3   Pass
     S0003   20          9.9                   37.1                5.2          5.8             Poor        68.9   Pass
     S0004   19          6.6                   30.0               11.3          3.9             Poor        44.5   Fail
     S0005   24          7.7                   59.0                7.3          5.3             Poor        57.6   Pass
     S0006   17          1.5                   30.0               14.7          3.0             Good        24.1   Fail
     S0007   25          5.9                   55.6               14.0          5.0          Average        48